# 03 · Whisper ASR —— 给机器装耳朵，听懂中英两段

**家族位置**：09 领域应用第 3 站（收官）。前两站是结构化数据（表格时序/评分矩阵），本章是真实连续信号：16kHz 波形 → log-Mel → tiny Transformer(39M) → 中英文字。

**学习目标**：音频→频谱→token 全链路；离线 tiny 推理；中英 WER 对照；强制语言/自动检测消融。

## 1. 原理：耳朵三件套

### 通俗理解

**一句话**：耳朵先把声波切成频谱图（看哪个频率响），再让小模型读图写字——英文像听写（词与词有空格），中文像听写连笔字（字与字没缝，按字算错）。

### 结构账

```
权重： whisper-tiny 本地 9 文件（model.safetensors 151MB + slow 分词 3 件套），local_files_only 离线
音频： 英 10.44s LibriSpeech（官方转写 28 词）/ 中 9.19s A7_87（参考 29 字）
链路： scipy读wav → processor转80维log-Mel → tiny(39,760,640参) generate → decode
口径： 英WER(词级) + 中WER(字级)；forced-lang vs auto 消融
```

In [ ]:
import os
os.environ['HF_HUB_OFFLINE']='1'; os.environ['TRANSFORMERS_OFFLINE']='1'
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.io import wavfile
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import random
random.seed(0)
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif']=['Microsoft YaHei','SimHei']
plt.rcParams['axes.unicode_minus']=False
W='weights/whisper-tiny'; D='data'
proc=WhisperProcessor.from_pretrained(W,local_files_only=True)
model=WhisperForConditionalGeneration.from_pretrained(W,local_files_only=True).eval()
print('tiny params:',sum(p.numel() for p in model.parameters()),flush=True)
def load_wav(p):
    sr,d=wavfile.read(p); return d.astype(np.float32)/32768.0,sr
xen,sen=load_wav(f'{D}/1089-134686-0000.wav'); xzh,szh=load_wav(f'{D}/A7_87.wav')
print(f'EN {len(xen)/sen:.2f}s@{sen}Hz / ZH {len(xzh)/szh:.2f}s@{szh}Hz',flush=True)
fig,ax=plt.subplots(2,1,figsize=(7,3.4))
ax[0].plot(xen[:16000]); ax[0].set_title('EN 波形(首1s)：朗读，包络平稳')
ax[1].plot(xzh[:16000]); ax[1].set_title('ZH 波形(首1s)：播报，停顿分明')
plt.tight_layout(); plt.savefig('figs/fig1_wave.png',dpi=150,bbox_inches='tight'); plt.show()

## 2. 转写：forced-lang 主结果 + auto 消融

In [ ]:
def transcribe(path,lang=None):
    x,sr=load_wav(path)
    f=proc(x,sampling_rate=sr,return_tensors='pt').input_features
    kw={} if lang is None else {'forced_decoder_ids':proc.get_decoder_prompt_ids(language=lang,task='transcribe')}
    with torch.no_grad(): out=model.generate(f,max_new_tokens=128,**kw)
    return proc.batch_decode(out,skip_special_tokens=True)[0].strip()
ref_en='HE HOPED THERE WOULD BE STEW FOR DINNER TURNIPS AND CARROTS AND BRUISED POTATOES AND FAT MUTTON PIECES TO BE LADLED OUT IN THICK PEPPERED FLOUR FATTENED SAUCE'
ref_zh=open(f'{D}/A7_87.txt',encoding='utf-8').read().strip()
h_en=transcribe(f'{D}/1089-134686-0000.wav','en'); h_zh=transcribe(f'{D}/A7_87.wav','zh')
a_en=transcribe(f'{D}/1089-134686-0000.wav'); a_zh=transcribe(f'{D}/A7_87.wav')
print('EN ref:',ref_en,flush=True); print('EN hyp:',h_en,flush=True)
print('ZH ref:',ref_zh,flush=True); print('ZH hyp:',h_zh,flush=True)
print('EN auto一致:',a_en==h_en,'| ZH auto一致:',a_zh==h_zh,flush=True)
feats=proc(xen,sampling_rate=sen,return_tensors='pt').input_features
print('log-Mel:',tuple(feats.shape),'(batch,80维,3000帧)',flush=True)
fig,ax=plt.subplots(figsize=(7,3.2)); ax.imshow(feats[0].numpy(),aspect='auto',origin='lower',cmap='magma')
ax.set_title('EN log-Mel 频谱图：模型真正读的图'); ax.set_xlabel('frames'); ax.set_ylabel('mel bins')
plt.tight_layout(); plt.savefig('figs/fig2_mel.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 打分：WER + 消融 + 总结

## 4. 总结与下一步（09 家族收官）

tiny 中英双转写 + WER + auto/forced 消融 + 频谱可视化。09 家族：时序(01)→推荐(02)→语音(03)，领域×主干三站收官。

In [ ]:
import numpy as np
def edit(a,b):
    d=np.zeros((len(a)+1,len(b)+1),dtype=int)
    for i in range(len(a)+1): d[i,0]=i
    for j in range(len(b)+1): d[0,j]=j
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            d[i,j]=min(d[i-1][j]+1,d[i][j-1]+1,d[i-1][j-1]+(a[i-1]!=b[j-1]))
    return d[len(a),len(b)]
wer_en=edit(ref_en.lower().split(),h_en.lower().split())/len(ref_en.split())
wer_zh=edit(list(ref_zh.replace(' ','')),list(''.join(h_zh.split())))/len(ref_zh.replace(' ',''))
print(f'EN WER={wer_en:.4f} | ZH WER(char)={wer_zh:.4f}',flush=True)
fig,ax=plt.subplots(figsize=(6,3.2))
ax.bar(['EN(词级)','ZH(字级)'],[wer_en,wer_zh],color=['#4C72B0','#C44E52'])
for i,v in enumerate([wer_en,wer_zh]): ax.text(i,v+0.02,f'{v:.4f}',ha='center')
ax.set_ylim(0,1.0); ax.set_ylabel('WER'); ax.set_title('tiny 中英 WER：英文可用，中文压力大')
plt.tight_layout(); plt.savefig('figs/fig3_wer.png',dpi=150,bbox_inches='tight'); plt.show()
print('SUMMARY',round(wer_en,4),round(wer_zh,4),a_en==h_en,a_zh==h_zh)